# Sovereign ModernBERT-Large: Proprietary Fine-Tuning Pipeline (V3 - Scaler-Free BFloat16)
### Training on 36,000 Balanced Clean Open-Pool Passages + Anchor Ground Truth

**Strategic Mission:** Break the 0.659 public ceiling by upgrading the underlying Transformer Backbone from the 2,699-sample manifold to a pristine 36,000-sample balanced manifold.
- **Base Architecture:** `answerdotai/ModernBERT-large` (28 layers, 1024 hidden size, 16 heads)
- **Initialization:** Pre-aligned weights from `dhruvtibarewal/nlm-enc-v5` for ultra-fast convergence.
- **Memory Optimization:** Gradient checkpointing enabled, micro-batch 16, accum 4 (effective 64), expandable_segments=True.
- **Numerical Stability:** Native BFloat16 autocast (eliminating FP16 GradScaler incompatibility on BFloat16 tensors).
- **Dataset:** 36,000 clean passages (2,000 per class) from `crtproduction/name-that-llm-open-pool` (70k junk purged) + Anchor `train.csv`.
- **Output:** Sovereign Checkpoint `/kaggle/working/sovereign_modernbert/`.

In [ ]:
import os, sys, glob, json, time, gc, math, shutil
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoConfig, AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from safetensors.torch import load_file, save_file
from sklearn.metrics import balanced_accuracy_score

def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = True

seed_everything(42)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Active device: {device}, Device count: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    print(f'GPU 0: {torch.cuda.get_device_name(0)}')


In [ ]:
print('=== Step 1: Locating Datasets ===')
pool_paths = glob.glob('/kaggle/input/**/ntl_open_pool.parquet', recursive=True)
assert pool_paths, 'ntl_open_pool.parquet not found in inputs!'
pool_path = pool_paths[0]
print(f'Found open pool: {pool_path}')

train_paths = glob.glob('/kaggle/input/**/train.csv', recursive=True)
assert train_paths, 'train.csv not found in inputs!'
train_path = train_paths[0]
print(f'Found competition train.csv: {train_path}')

enc_configs = glob.glob('/kaggle/input/**/encoder/config.json', recursive=True)
assert enc_configs, 'nlm-enc-v5 encoder config not found!'
enc_dir = os.path.dirname(enc_configs[0])
enc_root = os.path.dirname(enc_dir)
print(f'Found pretrained ModernBERT root: {enc_root}')

meta_file = os.path.join(enc_root, 'meta.json')
with open(meta_file, 'r') as f:
    meta_info = json.load(f)
LABELS = meta_info['labels']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}
print(f'Target 18 labels: {LABELS}')

df_pool = pd.read_parquet(pool_path)
print(f'Loaded raw open pool: {len(df_pool):,} rows')
if 'flag_gemini37_routing_risk' in df_pool.columns:
    df_pool = df_pool[df_pool['flag_gemini37_routing_risk'] == False]
if 'flag_near_train_prompt' in df_pool.columns:
    df_pool = df_pool[df_pool['flag_near_train_prompt'] == False]
print(f'Filtered pool (no routing risk, no leak): {len(df_pool):,} rows')

N_PER_CLASS = 2000
sampled = []
for model_name, grp in df_pool.groupby('model'):
    n = min(len(grp), N_PER_CLASS)
    sampled.append(grp.sample(n=n, random_state=42))
df_pool_balanced = pd.concat(sampled, ignore_index=True)
print(f'Balanced pool sample: {len(df_pool_balanced):,} rows (2,000 per class)')

df_train_orig = pd.read_csv(train_path)
print(f'Loaded train.csv: {len(df_train_orig):,} rows')

val_list = []
for model_name, grp in df_train_orig.groupby('model'):
    val_list.append(grp.sample(n=30, random_state=42))
df_val = pd.concat(val_list, ignore_index=True)[['text', 'model']]
val_texts_set = set(df_val['text'])

df_train_rem = df_train_orig[~df_train_orig['text'].isin(val_texts_set)]
df_train_final = pd.concat([
    df_pool_balanced[['text', 'model']],
    df_train_rem[['text', 'model']],
    df_train_rem[['text', 'model']]
], ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f'Final Training Set size: {len(df_train_final):,} rows')
print(f'Final Validation Set size: {len(df_val):,} rows')


In [ ]:
class ModernBERTClassifier(nn.Module):
    def __init__(self, encoder_path, num_classes=18, pool_type='mean_last'):
        super().__init__()
        self.config = AutoConfig.from_pretrained(encoder_path)
        self.encoder = AutoModel.from_pretrained(encoder_path, config=self.config)
        # CRITICAL MEMORY OPTIMIZATION: Gradient Checkpointing drops activation VRAM by 75%
        self.encoder.gradient_checkpointing_enable()
        self.pool_type = pool_type
        self.classifier = nn.Linear(self.config.hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        mask_expanded = attention_mask.unsqueeze(-1).float()
        pooled = (last_hidden * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1e-9)
        logits = self.classifier(pooled)
        return logits

print('Instantiating model and loading pretrained weights...')
tokenizer = AutoTokenizer.from_pretrained(enc_dir)
model = ModernBERTClassifier(enc_dir, num_classes=18).to(device)

head_path = os.path.join(enc_root, 'head.safetensors')
if os.path.exists(head_path):
    head_weights = load_file(head_path)
    model.classifier.weight.data.copy_(head_weights['weight'])
    model.classifier.bias.data.copy_(head_weights['bias'])
    print('Successfully initialized classifier head from pretrained head.safetensors!')
else:
    print('Initialized classifier head randomly.')

print(f'VRAM allocated after model load: {torch.cuda.memory_allocated(0)/(1024**2):.1f} MB')


In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=384):
        self.texts = list(texts)
        self.labels = [label2id[l] for l in labels]
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts, labels = zip(*batch)
    enc = tokenizer(list(texts), padding=True, truncation=True, max_length=384, return_tensors='pt')
    return enc['input_ids'], enc['attention_mask'], torch.tensor(labels, dtype=torch.long)

train_dataset = TextDataset(df_train_final['text'], df_train_final['model'], tokenizer)
val_dataset = TextDataset(df_val['text'], df_val['model'], tokenizer)

# SAFE BATCH SIZE = 16 (consumes ~4.5 GB VRAM with gradient checkpointing)
BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
print(f'Train Batches: {len(train_loader)}, Val Batches: {len(val_loader)}')


In [ ]:
EPOCHS = 2
ACCUM_STEPS = 4  # Effective batch size = 16 * 4 = 64
TOTAL_STEPS = (len(train_loader) // ACCUM_STEPS) * EPOCHS

optimizer_grouped_parameters = [
    {'params': [p for n, p in model.encoder.named_parameters()], 'lr': 2e-5, 'weight_decay': 0.01},
    {'params': [p for n, p in model.classifier.named_parameters()], 'lr': 1e-4, 'weight_decay': 0.01}
]
optimizer = torch.optim.AdamW(optimizer_grouped_parameters)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * TOTAL_STEPS), num_training_steps=TOTAL_STEPS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

# Note: BFloat16 has identical dynamic range to Float32, so GradScaler is neither needed nor supported.
print(f'Starting Fine-Tuning in Native BFloat16 for {EPOCHS} Epochs (Total Optimizer Steps: {TOTAL_STEPS})...')
best_balanced_acc = 0.0
save_dir = '/kaggle/working/sovereign_modernbert'
os.makedirs(os.path.join(save_dir, 'encoder'), exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    t0 = time.time()
    optimizer.zero_grad()
    
    for step, (input_ids, attention_mask, labels) in enumerate(train_loader):
        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels) / ACCUM_STEPS
            
        loss.backward()
        total_loss += loss.item() * ACCUM_STEPS
        
        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()
            
        if (step + 1) % 400 == 0:
            vram = torch.cuda.memory_allocated(0)/(1024**2)
            print(f'Epoch {epoch} | Step {step+1}/{len(train_loader)} | Loss: {total_loss / (step+1):.4f} | VRAM: {vram:.0f}MB | LR: {scheduler.get_last_lr()[0]:.2e}')
            
    # Evaluation Phase
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in val_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(input_ids, attention_mask)
            preds = logits.argmax(dim=-1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(labels.numpy())
            
    b_acc = balanced_accuracy_score(val_targets, val_preds)
    elapsed = time.time() - t0
    print(f'=== Epoch {epoch} Complete in {elapsed/60:.1f}m | Val Balanced Accuracy: {b_acc:.4f} ===')
    
    # Save Best Model
    if b_acc > best_balanced_acc:
        best_balanced_acc = b_acc
        print(f'--> New Best Balanced Accuracy: {b_acc:.4f}! Saving checkpoint...')
        model.encoder.save_pretrained(os.path.join(save_dir, 'encoder'))
        tokenizer.save_pretrained(os.path.join(save_dir, 'encoder'))
        save_file({
            'weight': model.classifier.weight.data.cpu(),
            'bias': model.classifier.bias.data.cpu()
        }, os.path.join(save_dir, 'head.safetensors'))
        with open(os.path.join(save_dir, 'meta.json'), 'w') as f:
            json.dump({
                'labels': LABELS,
                'max_len': 384,
                'pool': 'mean_last',
                'model': 'answerdotai/ModernBERT-large',
                'val_balanced_accuracy': float(b_acc),
                'epochs_trained': epoch
            }, f, indent=2)
            
print(f'\n*** Fine-Tuning Complete! Best Balanced Accuracy: {best_balanced_acc:.4f} ***')


In [ ]:
print('=== Step 3: Verifying Saved Checkpoint ===')
for root, dirs, files in os.walk(save_dir):
    for f in files:
        fp = os.path.join(root, f)
        sz_mb = os.path.getsize(fp) / (1024 * 1024)
        print(f'{os.path.relpath(fp, save_dir):35s} | {sz_mb:8.2f} MB')

print('\nCheckpoint is verified and ready for deployment in Champion V54!')
